# Kriol → English NMT

This notebook trains a Kriol → English translator using NLLB (facebook/nllb-200-distilled-600M) with the Hugging Face Trainer.

- Cleans and preprocesses pairs, caching a cleaned CSV to speed reruns
- Trains a single-GPU baseline and saves a `final/` checkpoint with HF artifacts and `.pth`
- Optional: back-translation plan and custom tokenizer scaffolding (placeholders)

References:
- NLLB model card: https://huggingface.co/facebook/nllb-200-distilled-600M
- Transformers Seq2Seq docs: https://huggingface.co/docs/transformers/en/tasks/translation


### Step 1 — Environment & imports

In [9]:
import os
import torch
from torch.utils.data import Dataset
import ftfy  # For fixing mojibake
import unicodedata

torch.set_float32_matmul_precision("high")

# Enable TF32 on supported GPUs (like finetune_rag_llm.ipynb)
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 enabled for better GPU performance")
except Exception:
    print("⚠️ TF32 not available on this system")

import pandas as pd
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoConfig,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer as Trainer,
    Seq2SeqTrainingArguments as TrainingArguments,
)

print(torch.__version__)
print(torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


TF32 enabled for better GPU performance
2.8.0+cu129
True


### Step 2 — Config

In [10]:

class CFG:
    # Model & paths
    MODEL_NAME = "facebook/nllb-200-distilled-600M"
    OUTPUT_DIR = "../model/"

    # Data
    DATA_FILE = "../data/train_data.xlsx"
    CLEAN_DATA_FILE = "../data/train_data_cleaned.csv"
    SRC_COL = "kriol"
    TGT_COL = "english"
    VAL_SIZE = 0.2
    SEED = 42

    # NLLB language tags (proxy Kriol as Tok Pisin for tokenizer)
    SRC_LANG = "tpi_Latn"
    TGT_LANG = "eng_Latn"

    # Preprocessing
    APPLY_ENGLISH_LID = True
    MAX_TOKENS = 512
    LEN_RATIO = 3.0
    STRIP_PUNCT_SRC = False
    STRIP_PUNCT_TGT = False

    # Cleaning control
    SKIP_CLEAN_IF_EXISTS = True

    # Training - OPTIMIZED FOR FULL GPU POWER
    NUM_EPOCHS = 5
    BATCH_SIZE = 16  # Increased for maximum GPU utilization
    LR = 3e-5
    MAX_LEN = 512
    DROPOUT = 0.2
    ATTENTION_DROPOUT = 0.1

    # Decoding
    BEAM_SIZE = 5
    LENGTH_PENALTY = 0.9
    EARLY_STOPPING = True

    # Decoding/generation extras
    GEN_MAX_NEW_TOKENS = 32

    # COMET
    COMET_MODEL = "Unbabel/wmt22-comet-da"
    COMET_BATCH = 32

    # Trainer args - OPTIMIZED FOR FULL GPU POWER
    WARMUP_RATIO = 0.1
    GRAD_ACCUM_STEPS = 2  # Optimized for maximum GPU utilization
    LABEL_SMOOTHING = 0.1
    LOGGING_STEPS = 25  # More frequent logging
    SAVE_STEPS = 500
    SAVE_TOTAL_LIMIT = 3
    REPORT_TO = "tensorboard"  # TensorBoard tracking like finetune_rag_llm.ipynb

    # Optimization
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 1.0
    LR_SCHEDULER = "cosine"

# VRAM-aware batch size adjustment
try:
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024**3)
        print(f"GPU VRAM: {vram_gb:.1f} GB")

        # Conservative mapping to avoid OOM during generation/eval
        if vram_gb >= 16:
            CFG.BATCH_SIZE = 32
            CFG.EVAL_BATCH_SIZE = 64
        elif vram_gb >= 12:
            CFG.BATCH_SIZE = 24
            CFG.EVAL_BATCH_SIZE = 48
        elif vram_gb >= 8:
            CFG.BATCH_SIZE = 16
            CFG.EVAL_BATCH_SIZE = 32
        else:
            CFG.BATCH_SIZE = 8
            CFG.EVAL_BATCH_SIZE = 16

        print(f"Optimized batch size: {CFG.BATCH_SIZE}")
        print(f"Optimized eval batch size: {CFG.EVAL_BATCH_SIZE}")
except Exception:
    CFG.EVAL_BATCH_SIZE = max(16, CFG.BATCH_SIZE * 2)
    pass

# Mixed precision - hardcoded BF16 for RTX 5060
CFG.BF16, CFG.FP16 = True, False

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)

print("✓ Configuration updated with copy version values")
print(f"  Model: {CFG.MODEL_NAME}")
print(f"  Epochs: {CFG.NUM_EPOCHS}")
print(f"  Batch size: {CFG.BATCH_SIZE}")
print(f"  Mixed precision: BF16={CFG.BF16}, FP16={CFG.FP16}")
print(f"  Beam size: {CFG.BEAM_SIZE}, Length penalty: {CFG.LENGTH_PENALTY}")


GPU VRAM: 8.0 GB
Optimized batch size: 8
Optimized eval batch size: 16
✓ Configuration updated with copy version values
  Model: facebook/nllb-200-distilled-600M
  Epochs: 5
  Batch size: 8
  Mixed precision: BF16=True, FP16=False
  Beam size: 5, Length penalty: 0.9


### Step 3 — Load data

In [ ]:
def load_parallel_data() -> pd.DataFrame:
    """Load parallel corpus with dictionary augmentation, intelligent caching, and validation."""
    
    # Try to load cached clean data first
    if os.path.exists(CFG.CLEAN_DATA_FILE):
        print(f"✓ Loading cached clean data from {CFG.CLEAN_DATA_FILE}")
        try:
            df = pd.read_csv(CFG.CLEAN_DATA_FILE, encoding="utf-8")
            print(f"  Cached data: {len(df)} pairs")
            return df
        except Exception as e:
            print(f"⚠ Warning: Could not load cached data: {e}")
            print("  Falling back to raw data...")
    
    # Load sentences (parallel data)
    print(f"📂 Loading raw sentences from {CFG.DATA_FILE}")
    try:
        if CFG.DATA_FILE.endswith('.xlsx'):
            sentences_df = pd.read_excel(CFG.DATA_FILE)
        elif CFG.DATA_FILE.endswith('.csv'):
            sentences_df = pd.read_csv(CFG.DATA_FILE, encoding="utf-8")
        else:
            raise ValueError(f"Unsupported file format: {CFG.DATA_FILE}")
    except Exception as e:
        raise FileNotFoundError(f"Could not load data file: {e}")
    
    # Validate columns
    if CFG.SRC_COL not in sentences_df.columns or CFG.TGT_COL not in sentences_df.columns:
        raise ValueError(f"Missing columns in sentences: expected {CFG.SRC_COL}, {CFG.TGT_COL}. Available: {list(sentences_df.columns)}")
    
    # Load dictionary (English to Kriol)
    dict_file = "Kriol Dictionary CSV.csv"  # Adjust path if needed
    print(f"📂 Loading dictionary from {dict_file}")
    dict_df = pd.read_csv(dict_file, encoding="utf-8")
    
    # Handle quoted phrases in dictionary (e.g., "baby, little" -> "baby little")
    dict_df['English'] = dict_df['English'].str.strip().str.replace(r'"(.*?),\s*(.*?)"', r'\1 \2', regex=True)
    
    # Reverse dictionary for Kriol-to-English (src = Kriol, tgt = English)
    dict_df = dict_df.rename(columns={'Kriol': CFG.SRC_COL, 'English': CFG.TGT_COL})
    
    # Combine sentences and reversed dictionary
    sentences_df = sentences_df[[CFG.TGT_COL, CFG.SRC_COL]].rename(columns={CFG.TGT_COL: CFG.SRC_COL, CFG.SRC_COL: CFG.TGT_COL})  # Swap for Kriol src, English tgt
    df = pd.concat([sentences_df, dict_df], ignore_index=True)
    
    # Basic validation and drop empty/duplicates early
    original_size = len(df)
    df = df.dropna()
    df = df[(df[CFG.SRC_COL].astype(str).str.strip() != "") & (df[CFG.TGT_COL].astype(str).str.strip() != "")]
    df = df.drop_duplicates()
    
    cleaned_size = len(df)
    print(f"  Original combined: {original_size} rows")
    print(f"  After basic cleaning: {cleaned_size} rows")
    print(f"  Removed: {original_size - cleaned_size} empty/invalid/duplicate rows")
    
    return df




### Step 4 — Clean and persist dataset (merged cleaner + execution)

In [15]:
# Updated Imports (add to top if not present)
try:
    import ftfy  # For fixing mojibake
    import unicodedata
except ImportError:
    print("Install ftfy for better text fixing: pip install ftfy")

# Step 3 — Load data (as function)
def load_parallel_data() -> pd.DataFrame:
    """Load parallel corpus with dictionary augmentation, intelligent caching, and validation."""
    
    # Try to load cached clean data first
    if os.path.exists(CFG.CLEAN_DATA_FILE):
        print(f"✓ Loading cached clean data from {CFG.CLEAN_DATA_FILE}")
        try:
            df = pd.read_csv(CFG.CLEAN_DATA_FILE, encoding="utf-8")
            print(f"  Cached data: {len(df)} pairs")
            return df
        except Exception as e:
            print(f"⚠ Warning: Could not load cached data: {e}")
            print("  Falling back to raw data...")
    
    # Load sentences (parallel data)
    print(f"📂 Loading raw sentences from {CFG.DATA_FILE}")
    try:
        if CFG.DATA_FILE.endswith('.xlsx'):
            sentences_df = pd.read_excel(CFG.DATA_FILE)
        elif CFG.DATA_FILE.endswith('.csv'):
            sentences_df = pd.read_csv(CFG.DATA_FILE, encoding="utf-8")
        else:
            raise ValueError(f"Unsupported file format: {CFG.DATA_FILE}")
    except Exception as e:
        raise FileNotFoundError(f"Could not load data file: {e}")
    
    # Validate columns
    if CFG.SRC_COL not in sentences_df.columns or CFG.TGT_COL not in sentences_df.columns:
        raise ValueError(f"Missing columns in sentences: expected {CFG.SRC_COL}, {CFG.TGT_COL}. Available: {list(sentences_df.columns)}")
    
    # Load dictionary (English to Kriol)
    dict_file = "Kriol Dictionary CSV.csv"  # Adjust path if needed
    print(f"📂 Loading dictionary from {dict_file}")
    dict_df = pd.read_csv(dict_file, encoding="utf-8")
    
    # Handle quoted phrases in dictionary (e.g., "baby, little" -> "baby little")
    dict_df['English'] = dict_df['English'].str.strip().str.replace(r'"(.*?),\s*(.*?)"', r'\1 \2', regex=True)
    
    # Reverse dictionary for Kriol-to-English (src = Kriol, tgt = English)
    dict_df = dict_df.rename(columns={'Kriol': CFG.SRC_COL, 'English': CFG.TGT_COL})
    
    # Combine sentences and reversed dictionary
    sentences_df = sentences_df[[CFG.TGT_COL, CFG.SRC_COL]].rename(columns={CFG.TGT_COL: CFG.SRC_COL, CFG.SRC_COL: CFG.TGT_COL})  # Swap for Kriol src, English tgt
    df = pd.concat([sentences_df, dict_df], ignore_index=True)
    
    # Basic validation and drop empty/duplicates early
    original_size = len(df)
    df = df.dropna()
    df = df[(df[CFG.SRC_COL].astype(str).str.strip() != "") & (df[CFG.TGT_COL].astype(str).str.strip() != "")]
    df = df.drop_duplicates()
    
    cleaned_size = len(df)
    print(f"  Original combined: {original_size} rows")
    print(f"  After basic cleaning: {cleaned_size} rows")
    print(f"  Removed: {original_size - cleaned_size} empty/invalid/duplicate rows")
    
    return df

# Step 4 — Clean data (as function)
def clean_parallel_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean parallel data with mojibake fixing, length checks, and optional punctuation stripping."""
    
    def fix_mojibake(text: str) -> str:
        """Fix common encoding artifacts like â€œ -> "."""
        try:
            text = unicodedata.normalize("NFC", ftfy.fix_text(text))
        except NameError:
            # Fallback manual replacements if ftfy not installed
            replacements = {
                'â€œ': '"', 'â€': '"', 'â€˜': "'", 'â€™': "'", 'â€¦': '...',
                'â€': '"', 'â€“': '-', 'â€”': '-', 'â€¢': '-', 'â„¢': '(TM)', 'â€š': ','
            }
            for wrong, right in replacements.items():
                text = text.replace(wrong, right)
        return text
    
    # Fix mojibake in both columns
    df[CFG.SRC_COL] = df[CFG.SRC_COL].astype(str).apply(fix_mojibake)
    df[CFG.TGT_COL] = df[CFG.TGT_COL].astype(str).apply(fix_mojibake)
    
    # Normalize whitespace (remove extra spaces, newlines)
    df[CFG.SRC_COL] = df[CFG.SRC_COL].str.replace(r'\s+', ' ', regex=True).str.strip()
    df[CFG.TGT_COL] = df[CFG.TGT_COL].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # Optional: Strip punctuation (kept as per your config)
    if CFG.STRIP_PUNCT_SRC:
        df[CFG.SRC_COL] = df[CFG.SRC_COL].str.replace(r'[^\w\s]', '', regex=True)
    if CFG.STRIP_PUNCT_TGT:
        df[CFG.TGT_COL] = df[CFG.TGT_COL].str.replace(r'[^\w\s]', '', regex=True)
    
    # Length checks (as per CFG)
    df['src_len'] = df[CFG.SRC_COL].str.split().str.len()
    df['tgt_len'] = df[CFG.TGT_COL].str.split().str.len()
    df = df[(df['src_len'] <= CFG.MAX_TOKENS) & (df['tgt_len'] <= CFG.MAX_TOKENS)]
    df = df[(df['src_len'] / df['tgt_len'] <= CFG.LEN_RATIO) & (df['tgt_len'] / df['src_len'] <= CFG.LEN_RATIO)]
    df = df.drop(columns=['src_len', 'tgt_len'])
    
    # Save cleaned data for caching
    df.to_csv(CFG.CLEAN_DATA_FILE, index=False, encoding='utf-8')
    print(f"✓ Cleaned data saved to {CFG.CLEAN_DATA_FILE} ({len(df)} pairs)")
    
    return df

# Step 5 — Split data (as function)
def split_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split into train/val with stratification for balance (e.g., by length)."""
    # Add temp length bin for stratification (to balance short/long sentences)
    df['len_bin'] = pd.qcut(df[CFG.SRC_COL].str.len(), q=5, labels=False)  # 5 bins by source length
    
    train_df, val_df = train_test_split(
        df, 
        test_size=CFG.VAL_SIZE, 
        random_state=CFG.SEED, 
        stratify=df['len_bin'] if len(df) > 1 else None  # Stratify if possible
    )
    train_df = train_df.drop(columns=['len_bin'])
    val_df = val_df.drop(columns=['len_bin'])
    
    print(f"✓ Split complete: Train={len(train_df)} pairs, Val={len(val_df)} pairs")
    return train_df, val_df

# Now call them in sequence (replace your original calls)
df = load_parallel_data()
df = clean_parallel_data(df)
train_df, val_df = split_data(df)

📂 Loading raw sentences from ../data/train_data.xlsx
📂 Loading dictionary from Kriol Dictionary CSV.csv


FileNotFoundError: [Errno 2] No such file or directory: 'Kriol Dictionary CSV.csv'

### Step 5 — Light normalization & data split

This step applies light normalization for better NLLB model performance:
- **English text**: Proper capitalization, punctuation, and grammar
- **Kriol text**: Consistent lowercase formatting, clean whitespace
- **NO artificial spacing** around punctuation (preserves natural flow)
- Remove duplicates and empty rows
- Simple train/validation split (80/20)

Note: This ensures grammatically correct English output while preserving natural Kriol characteristics.


In [8]:
def split_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split into train/val with stratification for balance (e.g., by length)."""
    # Add temp length bin for stratification (to balance short/long sentences)
    df['len_bin'] = pd.qcut(df[CFG.SRC_COL].str.len(), q=5, labels=False)  # 5 bins by source length
    
    train_df, val_df = train_test_split(
        df, 
        test_size=CFG.VAL_SIZE, 
        random_state=CFG.SEED, 
        stratify=df['len_bin'] if len(df) > 1 else None  # Stratify if possible
    )
    train_df = train_df.drop(columns=['len_bin'])
    val_df = val_df.drop(columns=['len_bin'])
    
    print(f"✓ Split complete: Train={len(train_df)} pairs, Val={len(val_df)} pairs")
    return train_df, val_df

### Step 6 — Tokenizer & Model

In [21]:
# Tokenizer & Model (default NLLB tokenizer)
src_lang = CFG.SRC_LANG
tgt_lang = CFG.TGT_LANG

print("Default NLLB tokenizer with language tags")
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME, src_lang=src_lang, tgt_lang=tgt_lang)
model = AutoModelForSeq2SeqLM.from_pretrained(CFG.MODEL_NAME)
if hasattr(model, "config"):
    model.config.use_cache = False
if hasattr(model, "config") and hasattr(tokenizer, "lang_code_to_id"):
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    if forced_bos is not None and forced_bos != tokenizer.unk_token_id:
        model.config.forced_bos_token_id = forced_bos
    if getattr(model.config, "decoder_start_token_id", None) is None:
        model.config.decoder_start_token_id = model.config.forced_bos_token_id

assert torch.cuda.is_available(), "CUDA is not available. Please check your GPU drivers and PyTorch install."
device = torch.device("cuda")
model.to(device)
print(device, torch.cuda.get_device_name(0))


Default NLLB tokenizer with language tags
cuda NVIDIA GeForce RTX 5060 Laptop GPU


### Step 7 — BitFit (Bias Tuning Only) for Kriol→English Translation


In [22]:
# 🔧 LoRA (Low-Rank Adaptation) - ~10% Trainable Parameters
# Uses low-rank matrices to adapt the model with controlled trainable parameters

from peft import get_peft_model, LoraConfig, TaskType
import torch

print("🔧 Setting up LoRA (Low-Rank Adaptation) for Kriol→English translation...")

# Step 1: Reload model fresh
model = AutoModelForSeq2SeqLM.from_pretrained(
    CFG.MODEL_NAME,
    dtype=torch.bfloat16 if CFG.BF16 else torch.float16 if CFG.FP16 else torch.float32,
    low_cpu_mem_usage=True
).to(device)

# Step 2: Set language config
tgt_lang_id = tokenizer.convert_tokens_to_ids(CFG.TGT_LANG)
model.config.decoder_start_token_id = tgt_lang_id
model.generation_config.forced_bos_token_id = tgt_lang_id

# Step 3: LoRA configuration for ~10% trainable parameters
print("🔧 Configuring LoRA for ~10% trainable parameters...")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=96,                    # Rank - increased to get closer to 10%
    lora_alpha=192,          # Scaling factor (typically 2x rank)
    lora_dropout=0.1,        # Dropout for LoRA layers
    target_modules=[         # Target modules to apply LoRA to
        "q_proj", "k_proj", "v_proj", "out_proj",  # Attention layers
        "fc1", "fc2",                              # Feed-forward layers
        "lm_head"                                  # Language modeling head
    ],
    bias="all",               # Train bias terms as well
    use_rslora=False,         # Use standard LoRA
    modules_to_save=None,     # No additional modules to save
)

# Step 4: Apply LoRA
model = get_peft_model(model, lora_config)

# Step 5: Verify setup
print("✅ LoRA applied successfully!")
model.print_trainable_parameters()

# Step 6: Test forward/backward pass
print("\n🧪 Testing LoRA model...")
try:
    # Disable cache to avoid issues
    model.config.use_cache = False
    
    dummy_inputs = tokenizer("test kriol text", return_tensors="pt").to(device)
    dummy_labels = tokenizer("test english text", return_tensors="pt").to(device)
    
    # Forward pass
    outputs = model(**dummy_inputs, labels=dummy_labels["input_ids"], use_cache=False)
    print(f"✅ Forward pass successful!")
    print(f"   Loss: {outputs.loss}")
    print(f"   Loss requires grad: {outputs.loss.requires_grad}")
    
    if outputs.loss.requires_grad:
        outputs.loss.backward()
        print("✅ Backward pass successful!")
        print("🎉 LoRA model is ready for training!")
    else:
        print("❌ No gradients - this shouldn't happen")
        
except Exception as e:
    print(f"❌ Error: {e}")

🔧 Setting up LoRA (Low-Rank Adaptation) for Kriol→English translation...
🔧 Configuring LoRA for ~10% trainable parameters...
✅ LoRA applied successfully!
trainable params: 76,932,416 || all params: 691,672,384 || trainable%: 11.122666999525602

🧪 Testing LoRA model...
✅ Forward pass successful!
   Loss: 12.3125
   Loss requires grad: True
✅ Backward pass successful!
🎉 LoRA model is ready for training!


### Step 11 — Dataset

In [23]:

class PairedTextDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer: AutoTokenizer, max_len: int):
        self.src = df[CFG.SRC_COL].tolist()
        self.tgt = df[CFG.TGT_COL].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx: int):
        src_text = str(self.src[idx])
        tgt_text = str(self.tgt[idx])
        model_inputs = self.tokenizer(
            src_text,
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        labels = self.tokenizer(
            text_target=tgt_text,
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in model_inputs.items()}
        item["labels"] = labels["input_ids"].squeeze(0)
        return item

train_ds = PairedTextDataset(train_df, tokenizer, CFG.MAX_LEN)
val_ds = PairedTextDataset(val_df, tokenizer, CFG.MAX_LEN)
len(train_ds), len(val_ds)


(18180, 4545)

### Step 12.1 — Trainer setup (DDP-ready) / Length-grouped sampler
Creates `LengthGroupedSampler` to cluster sequences of similar lengths, reducing padding and stabilizing training.


In [24]:
from transformers.trainer_pt_utils import LengthGroupedSampler
# Curriculum-like batching: length-grouped sampler to reduce padding and stabilize training
train_sampler = LengthGroupedSampler(lengths=[len(str(x).split()) for x in train_df[CFG.SRC_COL].tolist()],
                                     batch_size=CFG.BATCH_SIZE)



### Step 12.2 — Custom Trainer class
Defines `CleanSeq2SeqTrainer` that sanitizes inputs (drops unintended embed keys) to avoid HF arg conflicts.


In [25]:
# Utility: Trainer that drops unintended *_embeds keys to avoid HF arg conflicts
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

class CleanSeq2SeqTrainer(Seq2SeqTrainer):
    def _prepare_inputs(self, inputs):
        # Sanitize at input-prep stage too
        inputs.pop("decoder_inputs_embeds", None)
        inputs.pop("inputs_embeds", None)
        inputs.pop("decoder_input_ids", None)
        allowed = {"input_ids", "attention_mask", "labels", "decoder_attention_mask"}
        filtered = {k: v for k, v in inputs.items() if k in allowed}
        return super()._prepare_inputs(filtered)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Drop any embed keys and strictly whitelist safe args for seq2seq training
        inputs.pop("decoder_inputs_embeds", None)
        inputs.pop("inputs_embeds", None)
        inputs.pop("decoder_input_ids", None)
        allowed = {"input_ids", "attention_mask", "labels", "decoder_attention_mask"}
        filtered = {k: v for k, v in inputs.items() if k in allowed}
        # Call model explicitly with safe kwargs to avoid decoder ids/embeds conflicts
        outputs = model(
            decoder_input_ids=None,
            decoder_inputs_embeds=None,
            use_cache=False,
            **filtered,
        )
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
        return (loss, outputs) if return_outputs else loss



### Step 12.3 — Regularization
Applies dropout and attention-dropout settings on the model config for regularization.


In [26]:
# Regularization: set model-level dropouts if supported by config
if hasattr(model, "config"):
    model.config.use_cache = False
    if hasattr(model.config, "dropout"):
        model.config.dropout = CFG.DROPOUT
    if hasattr(model.config, "activation_dropout"):
        model.config.activation_dropout = CFG.DROPOUT
    if hasattr(model.config, "decoder_attention_dropout"):
        model.config.decoder_attention_dropout = CFG.ATTENTION_DROPOUT
    if hasattr(model.config, "encoder_attention_dropout"):
        model.config.encoder_attention_dropout = CFG.ATTENTION_DROPOUT



### Step 12.4 — Trainer args and instantiation
Builds `DataCollatorForSeq2Seq`, `TrainingArguments`, and instantiates `CleanSeq2SeqTrainer` with early stopping.


In [27]:
label_pad_token_id = -100
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, label_pad_token_id=label_pad_token_id, padding=True)

args = TrainingArguments(
    output_dir=CFG.OUTPUT_DIR,
    num_train_epochs=CFG.NUM_EPOCHS,
    per_device_train_batch_size=CFG.BATCH_SIZE,
    per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,  # optional: larger eval batch
    learning_rate=CFG.LR,
    gradient_accumulation_steps=CFG.GRAD_ACCUM_STEPS,
    label_smoothing_factor=CFG.LABEL_SMOOTHING,
    weight_decay=CFG.WEIGHT_DECAY,
    optim="adamw_torch",
    logging_steps=CFG.LOGGING_STEPS,
    save_steps=CFG.SAVE_STEPS,
    save_total_limit=CFG.SAVE_TOTAL_LIMIT,
    bf16=CFG.BF16,
    report_to=CFG.REPORT_TO,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    eval_accumulation_steps=1,
    remove_unused_columns=False,
    label_names=["labels"],
    group_by_length=True,

    # Advanced optimization
    lr_scheduler_type="cosine_with_restarts",
    warmup_ratio=0.1,

    # Gradient optimization
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    # Loss-only eval (no text generation)
    predict_with_generate=False,
)

trainer = CleanSeq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,            # or: processing_class=tokenizer (new API)
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

C:\Users\TARIK\AppData\Local\Temp\ipykernel_19536\3335375859.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CleanSeq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = CleanSeq2SeqTrainer(


### Step 13 — Train

In [28]:

trainer.train()

# Evaluate once to log metrics
metrics = trainer.evaluate()
print("eval_loss:", metrics.get("eval_loss"))

# Launch TensorBoard from notebook (like finetune_rag_llm.ipynb)
%load_ext tensorboard
%tensorboard --logdir "../model/tb"


c:\Users\TARIK\Desktop\Charles Darwin University\4 - Year 1 - Semester 2\IT CODE FAIR\AI Challenge\venv\Lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,2.768700,2.614552
2,2.757400,2.552427
3,2.703700,2.540778
4,2.708000,2.540106
5,2.659800,2.538247


c:\Users\TARIK\Desktop\Charles Darwin University\4 - Year 1 - Semester 2\IT CODE FAIR\AI Challenge\venv\Lib\site-packages\peft\utils\save_and_load.py:134: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
c:\Users\TARIK\Desktop\Charles Darwin University\4 - Year 1 - Semester 2\IT CODE FAIR\AI Challenge\venv\Lib\site-packages\torch\utils\checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
c:\Users\TARIK\Desktop\Charles Darwin University\4 - Year 1 - Semester 2\IT CODE FAIR\AI Challenge\venv\Lib\site-packages\peft\utils\save_and_load.py:134: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
c:\Users\TARIK\Deskto

eval_loss: 2.5386714935302734


Reusing TensorBoard on port 6006 (pid 21972), started 1:49:28 ago. (Use '!kill 21972' to kill it.)

### Step 14 — Save HF artifacts and a .pth checkpoint

In [29]:

final_dir = os.path.join(CFG.OUTPUT_DIR, "final")
os.makedirs(final_dir, exist_ok=True)
trainer.save_model(final_dir)
model_path = os.path.join(final_dir, "model_state.pth")
torch.save(model.state_dict(), model_path)
print(f"Saved .pth to: {model_path}")


c:\Users\TARIK\Desktop\Charles Darwin University\4 - Year 1 - Semester 2\IT CODE FAIR\AI Challenge\venv\Lib\site-packages\peft\utils\save_and_load.py:134: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Saved .pth to: ../model/final\model_state.pth


### Step 15 — Inference helper (final only)

In [30]:
final_dir = os.path.join(CFG.OUTPUT_DIR, "final")

final_tok = AutoTokenizer.from_pretrained(final_dir, src_lang=(CFG.SRC_LANG or "eng_Latn"), tgt_lang=(CFG.TGT_LANG or "eng_Latn"))
final_model = AutoModelForSeq2SeqLM.from_pretrained(final_dir).to(device)

@torch.no_grad()
def generate_with(model, tok, texts):
    inputs = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=CFG.MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    forced_bos = tok.convert_tokens_to_ids(CFG.TGT_LANG or "eng_Latn")
    out = model.generate(
        **inputs,
        max_new_tokens=CFG.GEN_MAX_NEW_TOKENS,
        num_beams=CFG.BEAM_SIZE,
        length_penalty=CFG.LENGTH_PENALTY,
        early_stopping=CFG.EARLY_STOPPING,
        forced_bos_token_id=forced_bos,
    )
    decoded = tok.batch_decode(out, skip_special_tokens=True)
    # Clean display artifacts (mojibake)
    try:
        from ftfy import fix_text as _fix_text
        import unicodedata as _ud
        decoded = [_ud.normalize("NFC", _fix_text(t)) for t in decoded]
    except Exception:
        pass
    return decoded

# Pick 3 random samples from training data and show Kriol / predicted / original
rows = val_df.sample(n=3)
kriols = rows[CFG.SRC_COL].astype(str).tolist()
eng_refs = rows[CFG.TGT_COL].astype(str).tolist()
eng_preds = generate_with(final_model, final_tok, kriols)

for i, (kriol, pred, ref) in enumerate(zip(kriols, eng_preds, eng_refs), start=1):
    print(f"Sample {i}")
    print("Kriol:", kriol)
    print("English predicted:", pred)
    print("English original:", ref)
    print("-")



Sample 1
Kriol: upset det king bean talk â€œyu report yu in, go talk completely into mi nearby en make mi kristjan streidawei?â€
English predicted: â€œYou report you in, go talk completely into me nearby and make me a Christian prostitute?â€.
English original: Agrippa said to Paul, â€œWith a little persuasion are you trying to make me a Christian?â€.
-
Sample 2
Kriol: en wen detlot streinja people hu sit mine enami you irrim mi talk they all day bradin into mi, en they all day gibap en kamat from det play where deibin hide yourselves
English predicted: 
English original: The foreigners will fade away, and will come trembling out of their close places.
-
Sample 3
Kriol: wen she save people from det play into dedbala en bring you into det appropriate play into sit laibalawan.
English predicted: Yes, she saves people from det play into detbala and brings you into det appropriate play into the labyrinth.
English original: To bring back his soul from the pit, that he may be enlightened w

### Step 16 — COMET evaluation (final only)
Scores validation translations with Unbabel COMET if available; otherwise prints a note (Python 3.13 may lack wheels).


In [ ]:
# COMET: evaluate final only
try:
    from comet import download_model, load_from_checkpoint

    BATCH = CFG.COMET_BATCH
    refs = val_df[CFG.TGT_COL].tolist()
    srcs = val_df[CFG.SRC_COL].tolist()

    # Show progress while generating hypotheses for COMET
    try:
        from tqdm import tqdm as _tqdm
        _iter = _tqdm(
            range(0, len(val_df), BATCH),
            total=(len(val_df) + BATCH - 1) // BATCH,
            desc="COMET prep: generating translations",
        )
    except Exception:
        _iter = range(0, len(val_df), BATCH)

    # Faster generation for COMET (greedy, mixed precision)
    @torch.no_grad()
    def fast_generate(texts):
        inputs = final_tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=CFG.MAX_LEN)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        forced_bos = final_tok.convert_tokens_to_ids(CFG.TGT_LANG or "eng_Latn")
        use_cuda = torch.cuda.is_available()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
            out = final_model.generate(
                **inputs,
                max_new_tokens=CFG.GEN_MAX_NEW_TOKENS,
                num_beams=1,
                do_sample=False,
                length_penalty=1.0,
                early_stopping=True,
                forced_bos_token_id=forced_bos,
            )
        return final_tok.batch_decode(out, skip_special_tokens=True)

    def batched_hyps_final():
        hyps = []
        for i in _iter:
            hyps.extend(fast_generate(srcs[i:i+BATCH]))
        return hyps

    hyps_final = batched_hyps_final()
    data_final = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(srcs, hyps_final, refs)]

    model_path = download_model(CFG.COMET_MODEL)
    comet_model = load_from_checkpoint(model_path)

    def get_score(output):
        if isinstance(output, dict):
            return output.get("system_score") or output.get("score") or output.get("mean_score")
        try:
            _, s = output
            return s
        except Exception:
            return output

    def get_segments(output):
        if isinstance(output, dict):
            segs = output.get("segments") or output.get("scores") or output.get("segment_scores")
            if isinstance(segs, list):
                return segs
        return None

    # Try to enable COMET's internal progress bar if supported
    try:
        out_final = comet_model.predict(
            data_final,
            batch_size=BATCH,
            gpus=1 if torch.cuda.is_available() else 0,
            progress_bar=True,
        )
    except TypeError:
        out_final = comet_model.predict(
            data_final,
            batch_size=BATCH,
            gpus=1 if torch.cuda.is_available() else 0,
        )

    sf = get_score(out_final)
    try:
        print("COMET (final):", f"{float(sf):.4f}")
    except Exception:
        print("COMET (final, raw):", sf)

    # Save COMET outputs to disk
    final_dir = os.path.join(CFG.OUTPUT_DIR, "final")
    os.makedirs(final_dir, exist_ok=True)

    def safe_float(x):
        try:
            return float(x)
        except Exception:
            return None

    # Write system score
    sf_f = safe_float(sf)
    try:
        with open(os.path.join(final_dir, "system_score.txt"), "w", encoding="utf-8") as f:
            f.write(f"{sf_f if sf_f is not None else sf}\n")
    except Exception as _e:
        print("Could not save final system score:", _e)

    # Write per-segment CSV (src, mt, ref, score)
    seg_final = get_segments(out_final)

    try:
        if isinstance(seg_final, list) and len(seg_final) == len(hyps_final):
            df_final = pd.DataFrame({
                "src": srcs,
                "mt": hyps_final,
                "ref": refs,
                "comet_score": seg_final,
            })
            df_final.to_csv(os.path.join(final_dir, "comet_segments.csv"), index=False)
    except Exception as _e:
        print("Could not save final segments CSV:", _e)

except Exception as e:
    print("COMET evaluation unavailable:", e)

